# Обработка geopqrquete файов для проверки скорости выполнения вычислений на batch

# Инициализация

In [1]:
import os
import sys

from sedona.spark import SedonaContext

# Явно указываем путь к Python для воркеров
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Настройка Hadoop
os.environ['HADOOP_HOME'] = r'C:\Hadoop\hadoop-3.3.6'

# Пакеты для Spark 3.5.4 со Scala 2.12
additional_packages = [
    "org.apache.sedona:sedona-spark-3.5_2.12:1.8.0",
    "org.datasyslab:geotools-wrapper:1.8.0-33.1"
]

config = SedonaContext.builder() \
    .appName("SedonaApp") \
    .config("spark.jars.packages", ",".join(additional_packages)) \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryo.registrator", "org.apache.sedona.core.serde.SedonaKryoRegistrator") \
    .config("spark.sql.extensions", "org.apache.sedona.sql.SedonaSqlExtensions,org.apache.sedona.viz.sql.SedonaVizExtensions") \
    .config("spark.jars", r"D:\Artem\Work\amtech_projects\postgresql-42.7.13.jar") \
    .master("local[*]") \
    .getOrCreate()

sedona = SedonaContext.create(config)


# Декоратор для замера скорости выполнения запроса

In [2]:
import time
from functools import wraps

def timer_sedona(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"Функция {func.__name__} выполнена за {end - start:.4f} секунд")
        return result
    return wrapper

# Функция для выполнения запросов

In [13]:
@timer_sedona
def execute_query(session, query, geoparquet_file_path):
    loaded_df = sedona.read.format("geoparquet").load(geoparquet_file_path)
    # new_df = loaded_df.selectExpr(query)
    loaded_df.createOrReplaceTempView("spatial_table")
    result_df = session.sql(query)
    result_df.show(10)
    # return result_df

# Пути к geoparquet файлам

In [14]:
features_1_path = r"D:\Artem\Work\amtech_projects\sedona_geoservice\data\features_1.geoparquet"
features_10_path = r"D:\Artem\Work\amtech_projects\sedona_geoservice\data\features_10.geoparquet"
features_100_path = r"D:\Artem\Work\amtech_projects\sedona_geoservice\data\features_100.geoparquet"
features_1k_path = r"D:\Artem\Work\amtech_projects\sedona_geoservice\data\features_1k.geoparquet"
features_10k_path = r"D:\Artem\Work\amtech_projects\sedona_geoservice\data\features_10k.geoparquet"
features_100k_path = r"D:\Artem\Work\amtech_projects\sedona_geoservice\data\features_100k.geoparquet"

# Обработка всех файлов сразу

In [17]:
def execute_query_all(sedona, sql_area):
    print("features_1_path: ")
    execute_query(sedona, sql_area, features_1_path)
    print("\nfeatures_10_path: ")
    execute_query(sedona, sql_area, features_10_path)
    print("\nfeatures_100_path: ")
    execute_query(sedona, sql_area, features_100_path)
    print("\nfeatures_1k_path: ")
    execute_query(sedona, sql_area, features_1k_path)
    print("\nfeatures_10k_path: ")
    execute_query(sedona, sql_area, features_10k_path)
    print("\nfeatures_100k_path: ")
    execute_query(sedona, sql_area, features_100k_path)
    

# Вычисление площади

In [18]:
# запрос
sql_area = """SELECT *, ST_Area(geometry) as area FROM spatial_table"""
execute_query_all(sedona, sql_area)
# execute_query(sedona, sql_area, features_1_path)
# execute_query(sedona, sql_area, features_10_path)
# execute_query(sedona, sql_area, features_100_path)
# execute_query(sedona, sql_area, features_1k_path)
# execute_query(sedona, sql_area, features_10k_path)
# execute_query(sedona, sql_area, features_100k_path)

features_1_path: 
+-------+--------+--------------------+-------------------+
|     id|layer_id|            geometry|               area|
+-------+--------+--------------------+-------------------+
|3724141|     170|MULTIPOLYGON (((3...|6.79783493526394E-8|
+-------+--------+--------------------+-------------------+

Функция execute_query выполнена за 0.1212 секунд

features_10_path: 
+-------+--------+--------------------+--------------------+
|     id|layer_id|            geometry|                area|
+-------+--------+--------------------+--------------------+
|3724141|     170|MULTIPOLYGON (((3...| 6.79783493526394E-8|
|3781881|     173|MULTIPOLYGON (((3...| 6.79783493526394E-8|
|3811204|     173|MULTIPOLYGON (((3...| 6.79783493526394E-8|
|3694818|     170|MULTIPOLYGON (((3...| 6.79783493526394E-8|
|3814588|     173|MULTIPOLYGON (((3...|2.771353487496294E-7|
|3692488|     170|MULTIPOLYGON (((3...|2.771353487496294E-7|
|3779551|     173|MULTIPOLYGON (((3...|2.771353487496294E-7|
|3

# Вычисление геометрия полигона в WKT формате

In [19]:
# запрос
sql_area = """SELECT *, ST_AsText(geometry) as geom_wkt FROM spatial_table"""
execute_query_all(sedona, sql_area)

features_1_path: 
+-------+--------+--------------------+--------------------+
|     id|layer_id|            geometry|            geom_wkt|
+-------+--------+--------------------+--------------------+
|3724141|     170|MULTIPOLYGON (((3...|MULTIPOLYGON (((3...|
+-------+--------+--------------------+--------------------+

Функция execute_query выполнена за 0.1316 секунд

features_10_path: 
+-------+--------+--------------------+--------------------+
|     id|layer_id|            geometry|            geom_wkt|
+-------+--------+--------------------+--------------------+
|3724141|     170|MULTIPOLYGON (((3...|MULTIPOLYGON (((3...|
|3781881|     173|MULTIPOLYGON (((3...|MULTIPOLYGON (((3...|
|3811204|     173|MULTIPOLYGON (((3...|MULTIPOLYGON (((3...|
|3694818|     170|MULTIPOLYGON (((3...|MULTIPOLYGON (((3...|
|3814588|     173|MULTIPOLYGON (((3...|MULTIPOLYGON (((3...|
|3692488|     170|MULTIPOLYGON (((3...|MULTIPOLYGON (((3...|
|3779551|     173|MULTIPOLYGON (((3...|MULTIPOLYGON (((3..

# Центроид полигона в WKT формате

In [20]:
sql_area = """SELECT *, ST_AsText(ST_Centroid(geometry)) as centroid_wkt FROM spatial_table"""
execute_query_all(sedona, sql_area)

features_1_path: 
+-------+--------+--------------------+--------------------+
|     id|layer_id|            geometry|        centroid_wkt|
+-------+--------+--------------------+--------------------+
|3724141|     170|MULTIPOLYGON (((3...|POINT (36.9926136...|
+-------+--------+--------------------+--------------------+

Функция execute_query выполнена за 0.1329 секунд

features_10_path: 
+-------+--------+--------------------+--------------------+
|     id|layer_id|            geometry|        centroid_wkt|
+-------+--------+--------------------+--------------------+
|3724141|     170|MULTIPOLYGON (((3...|POINT (36.9926136...|
|3781881|     173|MULTIPOLYGON (((3...|POINT (36.9926136...|
|3811204|     173|MULTIPOLYGON (((3...|POINT (36.9926136...|
|3694818|     170|MULTIPOLYGON (((3...|POINT (36.9926136...|
|3814588|     173|MULTIPOLYGON (((3...|POINT (36.9956353...|
|3692488|     170|MULTIPOLYGON (((3...|POINT (36.9956353...|
|3779551|     173|MULTIPOLYGON (((3...|POINT (36.9956353..

# Значение периметра полигона

In [23]:
sql_area = """SELECT *, ST_Perimeter(ST_Transform(ST_SetSRID(geometry, 4326), 'EPSG:3857')) as perimeter FROM spatial_table"""
execute_query_all(sedona, sql_area)

features_1_path: 
+-------+--------+--------------------+-----------------+
|     id|layer_id|            geometry|        perimeter|
+-------+--------+--------------------+-----------------+
|3724141|     170|MULTIPOLYGON (((3...|606.9581003871905|
+-------+--------+--------------------+-----------------+

Функция execute_query выполнена за 0.1137 секунд

features_10_path: 
+-------+--------+--------------------+------------------+
|     id|layer_id|            geometry|         perimeter|
+-------+--------+--------------------+------------------+
|3724141|     170|MULTIPOLYGON (((3...| 606.9581003871905|
|3781881|     173|MULTIPOLYGON (((3...| 606.9581003871905|
|3811204|     173|MULTIPOLYGON (((3...| 606.9581003871905|
|3694818|     170|MULTIPOLYGON (((3...| 606.9581003871905|
|3814588|     173|MULTIPOLYGON (((3...| 1533.054380913168|
|3692488|     170|MULTIPOLYGON (((3...|1533.0543809131677|
|3779551|     173|MULTIPOLYGON (((3...|1533.0543809131677|
|3727525|     170|MULTIPOLYGON (

# Координаты центроида полигона

In [24]:
sql_area = """
    with cenroid as (
        SELECT *, ST_Centroid(geometry) as centroid FROM spatial_table)
        select *, ST_X(centroid) as X_centroid, ST_Y(centroid) as Y_centroid FROM cenroid"""
execute_query_all(sedona, sql_area)

features_1_path: 
+-------+--------+--------------------+--------------------+-----------------+----------------+
|     id|layer_id|            geometry|            centroid|       X_centroid|      Y_centroid|
+-------+--------+--------------------+--------------------+-----------------+----------------+
|3724141|     170|MULTIPOLYGON (((3...|POINT (36.9926136...|36.99261364778196|55.1842351384604|
+-------+--------+--------------------+--------------------+-----------------+----------------+

Функция execute_query выполнена за 0.2005 секунд

features_10_path: 
+-------+--------+--------------------+--------------------+------------------+------------------+
|     id|layer_id|            geometry|            centroid|        X_centroid|        Y_centroid|
+-------+--------+--------------------+--------------------+------------------+------------------+
|3724141|     170|MULTIPOLYGON (((3...|POINT (36.9926136...| 36.99261364778196|  55.1842351384604|
|3781881|     173|MULTIPOLYGON (((3.

# Переставленные координаты центроида полигона

In [26]:
sql_area = """
    with flipped_cenroid_cord as (
        SELECT *, ST_FlipCoordinates(ST_Centroid(geometry)) as centroid FROM spatial_table)
        select *, ST_X(centroid) as X_centroid, ST_Y(centroid) as Y_centroid FROM flipped_cenroid_cord"""
execute_query_all(sedona, sql_area)

features_1_path: 
+-------+--------+--------------------+--------------------+----------------+-----------------+
|     id|layer_id|            geometry|            centroid|      X_centroid|       Y_centroid|
+-------+--------+--------------------+--------------------+----------------+-----------------+
|3724141|     170|MULTIPOLYGON (((3...|POINT (55.1842351...|55.1842351384604|36.99261364778196|
+-------+--------+--------------------+--------------------+----------------+-----------------+

Функция execute_query выполнена за 0.1081 секунд

features_10_path: 
+-------+--------+--------------------+--------------------+------------------+------------------+
|     id|layer_id|            geometry|            centroid|        X_centroid|        Y_centroid|
+-------+--------+--------------------+--------------------+------------------+------------------+
|3724141|     170|MULTIPOLYGON (((3...|POINT (55.1842351...|  55.1842351384604| 36.99261364778196|
|3781881|     173|MULTIPOLYGON (((3.

# Центроид полигона в WKT формате с переставленными координатами

In [28]:
sql_area = """
    with flipped_cenroid_cord (
        SELECT *, ST_FlipCoordinates(ST_Centroid(geometry)) as centroid FROM spatial_table)
        select *, ST_AsText(centroid) as flipped_centroid FROM flipped_cenroid_cord"""
execute_query_all(sedona, sql_area)

features_1_path: 
+-------+--------+--------------------+--------------------+--------------------+
|     id|layer_id|            geometry|            centroid|    flipped_centroid|
+-------+--------+--------------------+--------------------+--------------------+
|3724141|     170|MULTIPOLYGON (((3...|POINT (55.1842351...|POINT (55.1842351...|
+-------+--------+--------------------+--------------------+--------------------+

Функция execute_query выполнена за 0.1469 секунд

features_10_path: 
+-------+--------+--------------------+--------------------+--------------------+
|     id|layer_id|            geometry|            centroid|    flipped_centroid|
+-------+--------+--------------------+--------------------+--------------------+
|3724141|     170|MULTIPOLYGON (((3...|POINT (55.1842351...|POINT (55.1842351...|
|3781881|     173|MULTIPOLYGON (((3...|POINT (55.1842351...|POINT (55.1842351...|
|3811204|     173|MULTIPOLYGON (((3...|POINT (55.1842351...|POINT (55.1842351...|
|3694818| 

# Геометрия полигона с переставленными координатами в WKT формате

In [31]:
sql_area = """ SELECT *, ST_FlipCoordinates(geometry) as flipped_geoemtry FROM spatial_table"""
execute_query_all(sedona, sql_area)

features_1_path: 
+-------+--------+--------------------+--------------------+
|     id|layer_id|            geometry|    flipped_geoemtry|
+-------+--------+--------------------+--------------------+
|3724141|     170|MULTIPOLYGON (((3...|MULTIPOLYGON (((5...|
+-------+--------+--------------------+--------------------+

Функция execute_query выполнена за 0.1032 секунд

features_10_path: 
+-------+--------+--------------------+--------------------+
|     id|layer_id|            geometry|    flipped_geoemtry|
+-------+--------+--------------------+--------------------+
|3724141|     170|MULTIPOLYGON (((3...|MULTIPOLYGON (((5...|
|3781881|     173|MULTIPOLYGON (((3...|MULTIPOLYGON (((5...|
|3811204|     173|MULTIPOLYGON (((3...|MULTIPOLYGON (((5...|
|3694818|     170|MULTIPOLYGON (((3...|MULTIPOLYGON (((5...|
|3814588|     173|MULTIPOLYGON (((3...|MULTIPOLYGON (((5...|
|3692488|     170|MULTIPOLYGON (((3...|MULTIPOLYGON (((5...|
|3779551|     173|MULTIPOLYGON (((3...|MULTIPOLYGON (((5..

# Ближайшие объекты. 
## Находим ближайшие объекты к данному с помощью KNN

In [38]:
sql_area = """
WITH source_geometry AS (
    SELECT geometry
    FROM spatial_table
    WHERE id = 3724141 AND layer_id = 170
    LIMIT 1
)
SELECT 
    st.*,
    ST_Distance(so.geometry, st.geometry) as distance
FROM source_geometry so
JOIN spatial_table st 
ON ST_KNN(so.geometry, st.geometry, 50, false)
WHERE NOT (st.id = 3724141 AND st.layer_id = 170)  -- Исключаем исходный объект
ORDER BY distance
"""
execute_query_all(sedona, sql_area)

features_1_path: 
+---+--------+--------+--------+
| id|layer_id|geometry|distance|
+---+--------+--------+--------+
+---+--------+--------+--------+

Функция execute_query выполнена за 0.2508 секунд

features_10_path: 
+-------+--------+--------------------+--------------------+
|     id|layer_id|            geometry|            distance|
+-------+--------+--------------------+--------------------+
|3781881|     173|MULTIPOLYGON (((3...|                 0.0|
|3811204|     173|MULTIPOLYGON (((3...|                 0.0|
|3694818|     170|MULTIPOLYGON (((3...|                 0.0|
|3814588|     173|MULTIPOLYGON (((3...|1.047210986321356...|
|3727525|     170|MULTIPOLYGON (((3...|1.047210986321356...|
|3692488|     170|MULTIPOLYGON (((3...|1.047210986321356...|
|3779551|     173|MULTIPOLYGON (((3...|1.047210986321356...|
|3777619|     173|MULTIPOLYGON (((3...|0.003889468811908...|
|3729056|     170|MULTIPOLYGON (((3...|0.003889468811908...|
+-------+--------+--------------------+---------

# Упрощение геометрии объекта: ST_SimplifyPreserveTopology

In [40]:
sql_area = """ SELECT *, ST_SimplifyPreserveTopology(geometry, 10) as simplified_geoemtry FROM spatial_table"""
execute_query_all(sedona, sql_area)

features_1_path: 
+-------+--------+--------------------+--------------------+
|     id|layer_id|            geometry| simplified_geoemtry|
+-------+--------+--------------------+--------------------+
|3724141|     170|MULTIPOLYGON (((3...|POLYGON ((36.9938...|
+-------+--------+--------------------+--------------------+

Функция execute_query выполнена за 0.1331 секунд

features_10_path: 
+-------+--------+--------------------+--------------------+
|     id|layer_id|            geometry| simplified_geoemtry|
+-------+--------+--------------------+--------------------+
|3724141|     170|MULTIPOLYGON (((3...|POLYGON ((36.9938...|
|3781881|     173|MULTIPOLYGON (((3...|POLYGON ((36.9938...|
|3811204|     173|MULTIPOLYGON (((3...|POLYGON ((36.9938...|
|3694818|     170|MULTIPOLYGON (((3...|POLYGON ((36.9938...|
|3814588|     173|MULTIPOLYGON (((3...|POLYGON ((36.9937...|
|3692488|     170|MULTIPOLYGON (((3...|POLYGON ((36.9937...|
|3779551|     173|MULTIPOLYGON (((3...|POLYGON ((36.9937..

# Определить административную принадлежность объекта к округу и району
## Забираем актуальные границы оркгуов и районов из базы

In [45]:
credentials = dict({"host": "10.6.81.133", "port":"5432", "user":"postgres", "password":"tpY7H&sdvsdfsdf7zx9J"})
postgresql_url = f"jdbc:postgresql://{credentials.get('host')}:{credentials.get('port')}/mkgh_monitorings"

In [49]:
sql_get_regions = """select * from nsi.nsi_moscow_regions"""
df_regions = sedona.read \
    .format("jdbc") \
    .option("url", postgresql_url) \
    .option("user", f"{credentials.get('user')}") \
    .option("password", f"{credentials.get('password')}") \
    .option("dbtable", f"({sql_get_regions}) as subquery") \
    .option("driver", "org.postgresql.Driver") \
    .load()
df_regions.show(10)

+---+---------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------------------+
| id|region_id|           full_name|                name|short_name|       geometry_4326|       centroid_4326|              x_4326|              y_4326|       geometry_3857|       centroid_3857|              x_3857|              y_3857|         layer_alias|       creation_date|         start_date|           end_date|
+---+---------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------------------+
|  1| 11001200|Троицкий и Новомо...|Троицки

In [51]:
sql_get_districts = """select * from nsi.nsi_moscow_districts"""
df_districts = sedona.read \
    .format("jdbc") \
    .option("url", postgresql_url) \
    .option("user", f"{credentials.get('user')}") \
    .option("password", f"{credentials.get('password')}") \
    .option("dbtable", f"({sql_get_districts}) as subquery") \
    .option("driver", "org.postgresql.Driver") \
    .load()
df_districts.show(10)

+---+-----------+--------------------+--------------+------------------+---------+--------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------------------+
| id|district_id|           full_name|          name|        short_name|region_id|   region_name|region_short_name|       geometry_4326|       centroid_4326|              x_4326|              y_4326|       geometry_3857|       centroid_3857|              x_3857|              y_3857|         layer_alias|       creation_date|         start_date|           end_date|
+---+-----------+--------------------+--------------+------------------+---------+--------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------

In [64]:
df_districts.createOrReplaceTempView("districts_table")

sql_query = """
WITH source_objects AS (
    SELECT 
        id,
        layer_id,
        ST_Transform(ST_MakeValid(ST_SetSRID(geometry, 4326)), 'EPSG:3857') AS geometry_3857 
    FROM spatial_table
),
districts_transformed as (
    select  district_id, short_name, region_id, region_short_name,  ST_SetSRID(ST_GeomFromWKB(geometry_3857), 3857) as geometry_3857
    from districts_table
    ),
intersections_with_districts AS (
    SELECT 
        so.id,
        so.layer_id,
        md.district_id,
        md.short_name AS short_district_name,
        md.region_id,
        md.region_short_name AS region_short_name,
        ST_Area(ST_Intersection(so.geometry_3857, md.geometry_3857)) AS area_intersection
    FROM districts_transformed AS md
    CROSS JOIN source_objects AS so
    WHERE ST_Intersects(so.geometry_3857, md.geometry_3857)
),
ranked_intersections AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (PARTITION BY id, layer_id ORDER BY area_intersection DESC) AS rn
    FROM intersections_with_districts
)
SELECT 
    id,
    layer_id,
    district_id,
    short_district_name,
    region_id,
    region_short_name,
    area_intersection
FROM ranked_intersections
WHERE rn = 1
"""

execute_query_all(sedona, sql_query)

features_1_path: 
+-------+--------+-----------+-------------------+---------+-----------------+-----------------+
|     id|layer_id|district_id|short_district_name|region_id|region_short_name|area_intersection|
+-------+--------+-----------+-------------------+---------+-----------------+-----------------+
|3724141|     170|       1212|       р-н Вороново| 11001200|            ТиНАО|1475.445132269179|
+-------+--------+-----------+-------------------+---------+-----------------+-----------------+

Функция execute_query выполнена за 0.9101 секунд

features_10_path: 
+-------+--------+-----------+-------------------+---------+-----------------+------------------+
|     id|layer_id|district_id|short_district_name|region_id|region_short_name| area_intersection|
+-------+--------+-----------+-------------------+---------+-----------------+------------------+
|3692488|     170|       1212|       р-н Вороново| 11001200|            ТиНАО| 6015.049559528578|
|3694818|     170|       1212|     